# Modulo 6: Web Application & Dashboard Interattiva (Streamlit)
---
## Obiettivi del Modulo
Questo modulo affronta la trasformazione dei dati analitici (elaborati nel Modulo 5) in una **Web Application interattiva aziendale** per il monitoraggio in tempo reale di dati.

**Abilità e competenze sviluppate:**
1. Comprendere l'architettura di **Streamlit** e il suo modello di esecuzione reattivo (*Execution Model*).
2. Progettare Layout e componenti UI aziendali: filtri laterali (`st.sidebar`), carte metriche KPI (`st.metric`) ed elementi di selezione dinamica.
3. Sviluppare **Data Visualization dinamica e interattiva** con Plotly per il tracciamento temporale dei parametri (es. Nitrati, pH, Piombo) e dei relativi limiti normativi.
4. Implementare la **gestione visiva delle anomalie** (highlighting condizionale in rosso sui superamenti dei limiti di legge).
5. Assemblare ed erogare localmente una **Dashboard** pronta per l'uso aziendale.

Prima di iniziare, è necessario installare e verificare le librerie necessarie (`streamlit`, `plotly`, `pandas`, `numpy`).

In [ ]:
# [SETUP] Installazione ed importazione delle librerie
%pip install streamlit plotly pandas numpy

import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

print("Setup completato con successo!")
print("Versione Pandas:", pd.__version__)
print("Versione Streamlit:", st.__version__)

Il seguente script serve a generare il dataset unificato `referti_acque_clean.csv` (risultato della pipeline del Modulo 5), così da poter lavorare in modo totalmente autonomo.

In [ ]:
# [SETUP DATI DEMO] Generazione del dataset 'referti_acque_clean.csv'
import pandas as pd
import numpy as np

def genera_dataset_demo():
    np.random.seed(42)
    date_range = pd.date_range(start="2026-01-01", end="2026-03-15", freq="D")
    punti = ['PNT-01', 'PNT-02', 'PNT-03']
    laboratori = ['Alfa', 'Beta', 'Gamma']

    records = []
    for d in date_range:
        for p in punti:
            lab = np.random.choice(laboratori)

            # 1. pH (limite: 8.5)
            ph_val = round(np.random.normal(7.4, 0.6), 2)
            records.append({
                'data_prelievo': d.strftime('%Y-%m-%d'),
                'punto_prelievo': p,
                'laboratorio': lab,
                'parametro': 'pH',
                'valore': ph_val,
                'unita': 'pH',
                'limite': 8.5
            })

            # 2. Nitrati (limite: 50.0 mg/L)
            nitrati_val = round(np.random.normal(38.0, 10.0), 1)
            records.append({
                'data_prelievo': d.strftime('%Y-%m-%d'),
                'punto_prelievo': p,
                'laboratorio': lab,
                'parametro': 'Nitrati',
                'valore': max(2.0, nitrati_val),
                'unita': 'mg/L',
                'limite': 50.0
            })

            # 3. Piombo (limite: 10.0 ug/L)
            piombo_val = round(np.random.exponential(2.5), 1)
            records.append({
                'data_prelievo': d.strftime('%Y-%m-%d'),
                'punto_prelievo': p,
                'laboratorio': lab,
                'parametro': 'Piombo',
                'valore': piombo_val,
                'unita': 'ug/L',
                'limite': 10.0
            })

    df = pd.DataFrame(records)
    df.to_csv("referti_acque_clean.csv", index=False)
    print(f"Dataset demo generato con successo! {len(df)} righe salvate in 'referti_acque_clean.csv'")

genera_dataset_demo()

---
## 1: Introduzione a Streamlit & Modello di Esecuzione

* **Cos'è Streamlit?** È un framework open-source che consente di trasformare script di analisi dati Python in Web App interattive senza dover scrivere HTML, CSS o JavaScript.
* **Modello di Esecuzione (*Execution Model*):** A differenza dei tradizionali web framework (es. Flask o Django), Streamlit adotta un paradigma **reattivo**. Ogni volta che l'utente interagisce con un componente di input (es. muove uno slider o seleziona una voce nel menu), **Streamlit riesegue l'intero script Python dall'alto verso il basso**.
* **Lavorare in Jupyter / Colab con Streamlit:** Per sviluppare ed eseguire un'app Streamlit da un notebook, utilizziamo la magic command `%%writefile app.py` per salvare il codice in un file `.py` locale e poi lanciarlo dal terminale.

In [ ]:
%%writefile app_test_intro.py
# Prima semplice Web App Streamlit
import streamlit as st
import pandas as pd

st.title("🌊 Dashboard Acque - Test Iniziale")
st.write("Benvenuto nel modulo di visualizzazione ed analisi delle acque!")

# Caricamento dataset
df = pd.read_csv("referti_acque_clean.csv")

# Visualizzazione diretta del DataFrame
st.subheader("Anteprima del Dataset")
st.dataframe(df.head(10))

print("Script 'app_test_intro.py' creato con successo!")

> **💡 Come avviare l'App nel ambiente locale:**
> Aprire il terminale del sistema (o il terminale integrato in VS Code) e digitare:
> ```bash
> streamlit run app_test_intro.py
> ```
> L'interfaccia si aprirà automaticamente all'indirizzo `http://localhost:8501`.

### Creazione di un Tunnel

* **Il problema dell'ambiente Cloud:** Google Colab non viene eseguito sul una macchina locale, ma su una **macchina virtuale remota** nei server di Google.
* **Cosa succede con Streamlit:** Quando viene lanciato il comando `streamlit run`, l'applicazione si avvia sulla porta `8501` della macchina di Colab. Cercare di aprirla su `http://localhost:8501` dal proprio browser fallirà, perché il PC cercherà la pagina all'interno di se stesso e non nella macchina virtuale di Google.


> #### Il ruolo del Tunnel (`localtunnel` / `ngrok`):  
> Agisce come un "ponte sicuro": prende la porta interna `8501` della macchina remota di Colab e la espone su un **URL pubblico temporaneo**, permettendo di interagire con la Web App direttamente dal proprio browser.

In [ ]:
# 1. Ricava l'IP pubblico di Colab (serve come password per accedere al tunnel)
!curl ipv4.icanhazip.com

# 2. Avvia Streamlit in background e apri il tunnel sulla porta 8501
!streamlit run app_test_intro.py & npx localtunnel --port 8501

---
## 2: Layout UI, Filtri Laterali (Sidebar) e Carte Metriche (KPI)

Per creare dashboard professionali è fondamentale organizzare lo spazio:
- **`st.sidebar`**: Sposta i filtri (es. selezioni, date, menu a tendina) nella barra laterale sinistra per mantenere pulita la vista principale.
- **`st.columns`**: Suddivide l'area principale in colonne affiancate per mostrare informazioni sintetiche a colpo d'occhio.
- **`st.metric`**: Visualizza carte KPI (Key Performance Indicator) con titolo, valore ed eventuale variazione/stato del dato.

In [ ]:
%%writefile app_kpi_layout.py
import streamlit as st
import pandas as pd

st.set_page_config(page_title="KPI Acque", layout="wide")

df = pd.read_csv("referti_acque_clean.csv")

# --- SIDEBAR: BARRA LATERALE DI FILTRAGGIO ---
st.sidebar.header("🔍 Filtri di Ricerca")
punto_selezionato = st.sidebar.selectbox(
    "Seleziona Punto di Prelievo:",
    options=df['punto_prelievo'].unique()
)

# Filtraggio dei dati in base alla scelta
df_filtrato = df[df['punto_prelievo'] == punto_selezionato]

# --- AREA PRINCIPALE ---
st.title(f"📊 Monitoraggio: {punto_selezionato}")

# Calcolo KPI
totale_campioni = len(df_filtrato)
anomalie = df_filtrato[df_filtrato['valore'] > df_filtrato['limite']]
num_anomalie = len(anomalie)
pct_anomalie = (num_anomalie / totale_campioni * 100) if totale_campioni > 0 else 0.0

# Layout a 3 Colonne per le Carte Metriche
col1, col2, col3 = st.columns(3)

with col1:
    st.metric(label="Totale Analisi Effettuate", value=totale_campioni)

with col2:
    st.metric(label="Numero Anomalie (Fuori Norma)", value=num_anomalie, delta=num_anomalie, delta_color="inverse")

with col3:
    st.metric(label="Tasso Criticità (%)", value=f"{pct_anomalie:.1f}%")

### 💡 Micro-Challenge 1
**Obiettivo:** Creare uno script Streamlit (`app_mc1.py`) che aggiunga nella sidebar un **filtro multi-selezione (`st.sidebar.multiselect`)** per filtrare il dataset anche in base al **Laboratorio** di analisi (es. Alfa, Beta, Gamma).

*Suggerimento:* Usa `st.sidebar.multiselect("Seleziona Laboratori:", options=df['laboratorio'].unique(), default=df['laboratorio'].unique())` e applica il filtro con l'operatore `.isin()`.

In [ ]:
# === FAI QUI IL TUO ESPERIMENTO (Micro-Challenge 1) ===
%%writefile app_mc1.py
import streamlit as st
import pandas as pd

df = pd.read_csv("referti_acque_clean.csv")

st.sidebar.header("Filtri")
# 1. Selettore Multiplo Laboratori
labs_selezionati = st.sidebar.multiselect(
    "Seleziona Laboratori:",
    options=df['laboratorio'].unique(),
    default=df['laboratorio'].unique()
)

# 2. Applica il filtro con .isin()
df_filtrato = df[df['laboratorio'].isin(labs_selezionati)]

st.title("Risultati Filtrati per Laboratorio")
st.write(f"Record trovati: **{len(df_filtrato)}**")
st.dataframe(df_filtrato)

---
## 3: Data Visualization Dinamica con Plotly & Streamlit

I grafici statici non permettono di esplorare i dati nel dettaglio. Integrando **Plotly Express** con Streamlit tramite il comando `st.plotly_chart()`, otteniamo grafici interattivi con zoom, dati al passaggio del cursore (*tooltip*) e la possibilità di visualizzare il **limite di legge** sovrapposto all'andamento del parametro nel tempo.

In [ ]:
%%writefile app_grafici_plotly.py
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("referti_acque_clean.csv")
df['data_prelievo'] = pd.to_datetime(df['data_prelievo'])

st.title("📈 Analisi Temporale Parametri")

# Selettore parametro da analizzare
parametro_scelto = st.selectbox("Seleziona Parametro da Graficare:", options=df['parametro'].unique())
punto_scelto = st.selectbox("Seleziona Punto di Prelievo:", options=df['punto_prelievo'].unique())

# Filtro dati
df_graph = df[(df['parametro'] == parametro_scelto) & (df['punto_prelievo'] == punto_scelto)].sort_values('data_prelievo')

if not df_graph.empty:
    # Creazione grafico a linee Plotly
    fig = px.line(
        df_graph,
        x='data_prelievo',
        y='valore',
        markers=True,
        title=f"Andamento {parametro_scelto} nel tempo - {punto_scelto}",
        labels={'data_prelievo': 'Data Prelievo', 'valore': f"Valore ({df_graph['unita'].iloc[0]})"}
    )

    # Aggiunta linea orizzontale rossa per il Limite di Legge
    limite_val = df_graph['limite'].iloc[0]
    fig.add_hline(
        y=limite_val,
        line_dash="dash",
        line_color="red",
        annotation_text=f"Soglia Limite ({limite_val})",
        annotation_position="top right"
    )

    # Rendering in Streamlit
    st.plotly_chart(fig, use_container_width=True)
else:
    st.warning("Nessun dato disponibile per i filtri selezionati.")

### 💡 Micro-Challenge 2
**Obiettivo:** Modificare il grafico Plotly per evidenziare con **punti rossi** i singoli campionamenti che hanno **superato la soglia di legge**, lasciando in blu/grigio quelli nella norma.

*Suggerimento:* Crea una colonna booleana o categorica `Stato` nel DataFrame filtrato (`"ANOMALIA"` se `valore > limite` altrimenti `"OK"`), e passa l'argomento `color='Stato'` dentro `px.scatter()` o `px.line()`.

In [ ]:
# === FAI QUI IL TUO ESPERIMENTO (Micro-Challenge 2) ===
%%writefile app_mc2.py
import streamlit as st
import pandas as pd
import plotly.express as px

df = pd.read_csv("referti_acque_clean.csv")
df['Stato'] = df.apply(lambda r: 'ANOMALIA' if r['valore'] > r['limite'] else 'OK', axis=1)

# Grafico a dispersione con colori condizionali
fig = px.scatter(
    df[df['parametro'] == 'Nitrati'],
    x='data_prelievo',
    y='valore',
    color='Stato',
    color_discrete_map={'OK': 'green', 'ANOMALIA': 'red'},
    title="Nitrati: Evidenziazione Criticità"
)

st.plotly_chart(fig, use_container_width=True)

---
## 4: Evidenziazione Condizionale e Tabelle Interattive (Anomalie in Rosso)

Nelle tabelle gestionali aziendali è cruciale che l'operatore possa **individuare a colpo d'occhio i valori critici**.
Possiamo formattare lo stile del DataFrame Pandas prima di passarlo a `st.dataframe()` sfruttando la funzionalità `.style.apply()`.

In [ ]:
%%writefile app_table_highlight.py
import streamlit as st
import pandas as pd

df = pd.read_csv("referti_acque_clean.csv")

st.title("📋 Registro Analisi con Evidenziatore Anomalie")

# Funzione di formattazione condizionale per riga
def evidenzia_anomalie(row):
    if row['valore'] > row['limite']:
        return ['background-color: #ffcccc; color: darkred; font-weight: bold;'] * len(row)
    return [''] * len(row)

# Applicazione dello stile al DataFrame
df_styled = df.head(20).style.apply(evidenzia_anomalie, axis=1)

st.dataframe(df_styled, use_container_width=True)

### 💡 Micro-Challenge 3
**Obiettivo:** Invece di colorare l'intera riga, evidenzia **soltanto la cella della colonna `valore`** quando supera il `limite`.

*Suggerimento:* Definisci una funzione applicabile con `.style.apply()` a livello di colonne o applica la formattazione mirata con la logica `axis=1`.

In [ ]:
# === FAI QUI IL TUO ESPERIMENTO (Micro-Challenge 3) ===
%%writefile app_mc3.py
import streamlit as st
import pandas as pd

df = pd.read_csv("referti_acque_clean.csv")

def colora_solo_valore(row):
    styles = [''] * len(row)
    val_idx = row.index.get_loc('valore')
    if row['valore'] > row['limite']:
        styles[val_idx] = 'background-color: #ff4d4d; color: white; font-weight: bold;'
    return styles

st.dataframe(df.head(20).style.apply(colora_solo_valore, axis=1), use_container_width=True)

---
## 5: Assemblaggio della Dashboard Acque v1.0 (Web App Completa)

Uniamo tutti i componenti sviluppati finora in uno script completo ed elegante: `dashboard_acque.py`.

In [ ]:
%%writefile dashboard_acque.py
import streamlit as st
import pandas as pd
import plotly.express as px

# 1. Configurazione della Pagina Web
st.set_page_config(
    page_title="Dashboard Monitoraggio Acque",
    page_icon="💧",
    layout="wide"
)

# 2. Caricamento Dati (con Cache per Ottimizzare le Prestazioni)
@st.cache_data
def load_data():
    data = pd.read_csv("referti_acque_clean.csv")
    data['data_prelievo'] = pd.to_datetime(data['data_prelievo'])
    return data

df = load_data()

# 3. Header Principale
st.title("💧 Dashboard Interattiva - Quality Check Acque")
st.markdown("Sistema centralizzato per il monitoraggio dei parametri chimico-fisici e gestione anomalie normative.")
st.markdown("---")

# 4. Sidebar: Filtri di Controllo
st.sidebar.header("🎛️ Filtri Dashboard")
punti_disponibili = ["TUTTI"] + list(df['punto_prelievo'].unique())
punto_sel = st.sidebar.selectbox("Punto di Prelievo:", punti_disponibili)

parametro_sel = st.sidebar.selectbox("Parametro Chimico:", df['parametro'].unique())

# Applicazione Filtri
df_filtered = df.copy()
if punto_sel != "TUTTI":
    df_filtered = df_filtered[df_filtered['punto_prelievo'] == punto_sel]

# 5. Sezione KPI Cards
totale_record = len(df_filtered)
df_anomalie = df_filtered[df_filtered['valore'] > df_filtered['limite']]
num_anomalie = len(df_anomalie)
pct_anomalie = (num_anomalie / totale_record * 100) if totale_record > 0 else 0.0

kpi1, kpi2, kpi3 = st.columns(3)
kpi1.metric("Totale Campionamenti", totale_record)
kpi2.metric("Anomalie Rilevate", num_anomalie, delta=num_anomalie, delta_color="inverse")
kpi3.metric("Percentuale Fuori Norma", f"{pct_anomalie:.2f}%")

st.markdown("---")

# 6. Visualizzazione Grafica Dinamica
st.subheader(f"📈 Andamento Temporale: {parametro_sel}")
df_param = df_filtered[df_filtered['parametro'] == parametro_sel].sort_values('data_prelievo')

if not df_param.empty:
    fig = px.line(
        df_param,
        x='data_prelievo',
        y='valore',
        color='punto_prelievo',
        markers=True,
        labels={'data_prelievo': 'Data', 'valore': f"Esito ({df_param['unita'].iloc[0]})"}
    )
    # Aggiungi Soglia
    limite_val = df_param['limite'].iloc[0]
    fig.add_hline(y=limite_val, line_dash="dash", line_color="red", annotation_text="Soglia di Legge")
    st.plotly_chart(fig, use_container_width=True)
else:
    st.info("Nessun dato da mostrare per le impostazioni correnti.")

# 7. Tabella di Dettaglio con Anomalie Evidenziate
st.subheader("🚨 Registro dei Valori ed Anomalie")

def style_anomalies(row):
    if row['valore'] > row['limite']:
        return ['background-color: #ffcccc; color: #990000; font-weight: bold;'] * len(row)
    return [''] * len(row)

st.dataframe(df_filtered.style.apply(style_anomalies, axis=1), use_container_width=True)

print("Dashboard Acque v1.0 assemblata ed impacchettata in 'dashboard_acque.py'!")

---
## Esercitazione Pratica (1h)

### Task Aziendale:
Estendi la web app `dashboard_acque.py` creando una nuova versione chiamata `dashboard_acque_v2.py` aggiungendo **due nuove funzionalità professionali**:

1. **Pulsante di Download (`st.download_button`):** Nella sidebar, permetti all'utente di scaricare in formato **CSV** i soli dati filtrati visibili a schermo.
2. **Tabella Aggregata di Sintesi per Punto di Prelievo:** Aggiungi in fondo alla pagina una sezione che calcola e mostra le statistiche aggregate per ogni punto di prelievo (usando `df.groupby('punto_prelievo')`), calcolando: **Valore Medio**, **Valore Massimo** e **Numero di Anomalie Totali**.

<details>
<summary><b>Clicca qui per visualizzare la Soluzione dell'Esercitazione</b></summary>

```python
# === SOLUZIONE ESERCITAZIONE PRATICA ===
%%writefile dashboard_acque_v2.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Dashboard Acque v2.0", layout="wide")

@st.cache_data
def load_data():
    data = pd.read_csv("referti_acque_clean.csv")
    data['data_prelievo'] = pd.to_datetime(data['data_prelievo'])
    return data

df = load_data()

st.title("💧 Dashboard Acque v2.0 - con Analytics & Export")

# --- SIDEBAR CON DOWNLOAD BUTTON ---
st.sidebar.header("🎛️ Filtri & Export")
punto_sel = st.sidebar.selectbox("Punto di Prelievo:", ["TUTTI"] + list(df['punto_prelievo'].unique()))
parametro_sel = st.sidebar.selectbox("Parametro:", df['parametro'].unique())

df_filtered = df.copy()
if punto_sel != "TUTTI":
    df_filtered = df_filtered[df_filtered['punto_prelievo'] == punto_sel]

# NUOVA FEATURE 1: Download Button per i dati filtrati
csv_data = df_filtered.to_csv(index=False).encode('utf-8')
st.sidebar.download_button(
    label="📥 Scarica Dati Filtrati (CSV)",
    data=csv_data,
    file_name=f"export_acque_{punto_sel}.csv",
    mime="text/csv"
)

# --- GRAFICO TEMPORALE ---
df_param = df_filtered[df_filtered['parametro'] == parametro_sel].sort_values('data_prelievo')
if not df_param.empty:
    fig = px.line(df_param, x='data_prelievo', y='valore', color='punto_prelievo', markers=True)
    fig.add_hline(y=df_param['limite'].iloc[0], line_dash="dash", line_color="red")
    st.plotly_chart(fig, use_container_width=True)

# NUOVA FEATURE 2: Tabella Aggregata di Sintesi per Punto di Prelievo
st.markdown("---")
st.subheader("📊 Aggregazione Statistica per Punto di Prelievo")

# Calcolo Flag Anomalia per la groupby
df['is_anomalia'] = df['valore'] > df['limite']

df_summary = df.groupby('punto_prelievo').agg(
    Totale_Campionamenti=('valore', 'count'),
    Valore_Medio=('valore', 'mean'),
    Valore_Massimo=('valore', 'max'),
    Totale_Anomalie=('is_anomalia', 'sum')
).reset_index()

# Arrotondamento per pulizia estetica
df_summary['Valore_Medio'] = df_summary['Valore_Medio'].round(2)
df_summary['Valore_Massimo'] = df_summary['Valore_Massimo'].round(2)

st.dataframe(df_summary, use_container_width=True)
```
</details>